# Real vs. Synthetic vs. Fake Data

In this tutorial, we compare three types of data: **real**, **fake**, and **synthetic**. 
Using a credit card transaction dataset, we explore how each type differs in statistical 
quality and downstream performance.

We use the **DayZSynthesizer** to generate fake data (random values drawn from user-defined 
ranges) and the **GaussianCopulaSynthesizer** to generate synthetic data (values learned from 
the real data's statistical patterns).

**Use case:** You need to share a credit card dataset with a third-party vendor for testing, 
but the real data is too sensitive. Should you use randomly generated fake data or 
statistically modeled synthetic data? This tutorial answers that question.

## Loading and Exploring the Data

We use a credit card transaction dataset that contains over **24 million transactions** 
from **2,000 US-based consumers**, each with multiple credit cards. It includes:

- **Transaction info:** amount, timestamp (year, month, day, time), and payment method 
(chip, swipe, online)
- **Merchant info:** name, city, state, zip code, and category code (MCC)
- **Fraud label:** whether each transaction is fraudulent

Since the full dataset is large, we take a 10% random sample to use throughout this tutorial.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sdv.datasets.demo import download_demo

real_data, metadata = download_demo(
    modality='single_table',
    dataset_name='credit_card_transactions'
)

The `Amount` column is stored as a string with a dollar sign, and `Zip` has trailing 
decimals. Let's clean these up so they're easier to work with.

In [12]:
real_data["Amount"] = (
    real_data["Amount"].str.replace("$", "").astype(float)
)
real_data["Zip"] = (
    real_data["Zip"].astype(str)
    .str.replace(".0", "", regex=False)
)

real_data.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,1912,3,2016,12,25,10:32,33.79,Online Transaction,-6458444334611773637,ONLINE,NaN,nan,4784,NaN,No
1,1178,0,2012,12,19,09:17,36.64,Swipe Transaction,3635551857898739641,Doylestown,OH,44230,5310,NaN,No
2,330,0,2015,6,27,12:40,41.08,Chip Transaction,-1847597064923709092,West Newton,MA,2465,4121,NaN,No
3,1286,1,2013,1,16,13:29,100.00,Swipe Transaction,-4282466774399734331,Marion,IA,52302,4829,NaN,No
4,1543,2,2007,1,12,16:20,9.29,Swipe Transaction,-727612092139916043,Chicago,IL,60657,5411,NaN,No


**How large is the dataset and how many unique users does it have?**

In [13]:
print(f'Rows: {len(real_data):,}')
print(f'Columns: {real_data.shape[1]}')
print(f'Unique users: {real_data["User"].nunique():,}')

Full dataset: 24,386,900 rows, 15 columns
Sample dataset: 2,438,690 rows, 15 columns
Unique users (sample): 1,998
Unique users (full): 2,000


**What does the distribution of transaction amounts look like?**

*The vast majority of transactions are small — clustered near $0 — with a long 
right tail extending past $6,000. This heavy skew is typical of real-world spending 
behavior, where everyday purchases dominate and large transactions are rare.*

Breaking it down into ranges reveals more detail:

*Negative amounts (refunds) peak around -$100. The $0–$250 range shows a steep 
decline from very small purchases, with periodic spikes. Transactions above $250 
are sparse, forming a long tail out to $6,000+.*

**What years are represented in this dataset?**

*Transaction volume grows steadily from the early 1990s through 2018, reflecting 
increasing credit card adoption over time. There is a slight drop in 2020.*

**How balanced is the fraud label?**

*The dataset is highly imbalanced — the overwhelming majority of transactions are 
legitimate, with fraudulent transactions making up less than 1%. This class imbalance 
is realistic and makes the dataset a good test for synthetic data generators.*

**How do the key categorical columns relate to each other?**

The chart below shows the flow between payment method, transaction errors, and fraud status.

*Swipe transactions are the most common payment method, followed by chip and online. 
Most transactions have no errors. Fraud is rare across all payment methods, but 
occurs in every category — there is no single payment type that is fraud-free.*

## Creating the Metadata

SDV uses a **Metadata** object to understand the structure of your data — which columns are 
numerical, categorical, datetime, and so on.

Since we downloaded the dataset using `download_demo`, the metadata is already provided. 
However, the auto-detected types may not always be accurate. Let's update the columns 
that need correction.

In [ ]:
metadata.update_column(column_name="User", sdtype="categorical")
metadata.update_column(
    column_name="Year", sdtype="datetime",
    datetime_format="%Y"
)
metadata.update_column(
    column_name="Month", sdtype="datetime",
    datetime_format="%m"
)
metadata.update_column(
    column_name="Day", sdtype="datetime",
    datetime_format="%d"
)
metadata.update_column(
    column_name="Time", sdtype="datetime",
    datetime_format="%H:%M"
)
metadata.update_column(
    column_name="Merchant Name", sdtype="categorical"
)
metadata.update_column(
    column_name="Merchant City", sdtype="categorical"
)
metadata.update_column(
    column_name="Merchant State", sdtype="categorical"
)
metadata.update_column(column_name="MCC", sdtype="categorical")
metadata.update_column(column_name="Zip", sdtype="categorical")
metadata.update_column(
    column_name="Is Fraud?", sdtype="categorical"
)

metadata.visualize()

## Generating Fake Data with DayZSynthesizer

The **DayZSynthesizer** generates fake data by randomly sampling values from user-defined 
ranges and categories. Unlike a statistical model, it does not learn any patterns or 
correlations from the real data.

To make the fake data as realistic as possible, we configure it with the correct categories, 
numerical bounds, and null rates from the real dataset.

In [ ]:
from sdv.single_table import DayZSynthesizer

fake_synthesizer = DayZSynthesizer(metadata)

# Set missing value proportions to match real data
for col in ['Merchant State', 'Errors?']:
    fake_synthesizer.set_missing_values(
        column_name=col,
        missing_values_proportion=round(
            real_data[col].isnull().sum()
            / len(real_data),
            10
        )
    )

# Set valid categories for each categorical column
cat_columns = [
    'User', 'Card', 'Use Chip', 'Merchant Name',
    'Merchant City', 'Merchant State', 'Zip', 'MCC',
    'Errors?', 'Is Fraud?'
]
for col in cat_columns:
    fake_synthesizer.set_category_values(
        column_name=col,
        category_values=(
            real_data[col].dropna().unique().tolist()
        )
    )

# Set numerical bounds
fake_synthesizer.set_numerical_bounds(
    column_name='Amount',
    min_value=min(real_data['Amount']),
    max_value=max(real_data['Amount'])
)

# Set datetime bounds
for col in ['Year', 'Month', 'Day', 'Time']:
    fake_synthesizer.set_datetime_bounds(
        column_name=col,
        start_timestamp=str(real_data[col].min()),
        end_timestamp=str(real_data[col].max())
    )

Now let's generate fake data with the same number of rows as our real dataset.

In [ ]:
num_sample_rows = len(real_data)
fake_data = fake_synthesizer.sample(num_sample_rows)
fake_data.head()

## Generating Synthetic Data with GaussianCopulaSynthesizer

The **GaussianCopulaSynthesizer** learns the statistical distributions and correlations in 
the real data, then generates new rows that preserve those patterns. This is the key 
difference from fake data — synthetic data is *modeled*, not randomized.

In [ ]:
from sdv.single_table import GaussianCopulaSynthesizer

synthetic_synthesizer = GaussianCopulaSynthesizer(metadata)
synthetic_synthesizer.fit(real_data)

synthetic_data = synthetic_synthesizer.sample(num_sample_rows)
synthetic_data.head()

## Evaluating Data Quality

SDV provides a built-in quality evaluation that scores how well the generated data matches 
the real data's statistical properties. Scores range from 0 to 1, where 1 means a perfect 
statistical match.

Let's compare both the fake and synthetic datasets against the real data.

**How does the fake data compare to the real data?**

In [ ]:
from sdv.evaluation.single_table import evaluate_quality

evaluate_quality(
    real_data=real_data,
    synthetic_data=fake_data,
    metadata=metadata
)

**How does the synthetic data compare?**

In [ ]:
evaluate_quality(
    real_data=real_data,
    synthetic_data=synthetic_data,
    metadata=metadata
)

> **Key Takeaway:** Synthetic data achieves a significantly higher quality score than fake data because it learns the statistical patterns in your real data rather than generating random values.

## Comparing Transaction Amount Distributions

The quality score gives us an overall number, but let's look at specific columns to 
understand *where* the differences are. We break down transaction amounts into five 
fine-grained ranges and compare the distributions across all three datasets.

**Fine Grained Distributions for Real Transaction Amounts**

*The real data shows distinct patterns in each range: refunds cluster around -$100, 
small purchases ($1–$10) are extremely common, the $10–$100 range has a steep 
decline with periodic spikes at round-dollar amounts, $100–$500 drops off sharply, 
and transactions above $500 are rare with a long tail.*

**Fine Grained Distributions for Fake Transaction Amounts**

**Fine Grained Distributions for Fake Transaction Amounts**

*The fake data produces nearly flat, uniform distributions across every range. 
The characteristic peaks and patterns from the real data are completely absent — 
the $500+ range even shows values in the micro-range (150µ), far from realistic.*

**Fine Grained Distributions for Synthetic Transaction Amounts**

**Fine Grained Distributions for Synthetic Transaction Amounts**

*The synthetic data reproduces the overall shape of each range — the refund cluster, 
the steep decline in small purchases, and the right skew in higher amounts. 
While individual spikes are smoothed out, the general patterns are preserved.*

The synthetic data closely mirrors the shape of the real data's distributions across 
all amount ranges. The fake data, by contrast, shows flat uniform distributions that 
don't capture any of the real spending patterns.

## Simulating a Fraud Detection API

Quality scores and distribution plots tell us about statistical similarity, but do they 
translate to real-world performance? Let's test with a practical scenario.

We simulate a fraud detection function where runtime depends on the transaction amount — 
larger transactions take longer to process. If synthetic data preserves the amount 
distribution, the runtime behavior should match the real data.

In [ ]:
import numpy as np
import time

# Amount ranges and their corresponding (mean_runtime, std) values
ranges = [
    ((-float('inf'), 5), (0.1, 0.01)),
    ((5, 20), (0.2, 0.01)),
    ((20, 40), (0.3, 0.05)),
    ((40, 80), (0.5, 0.05)),
    ((80, float('inf')), (0.7, 0.05))
]


def calculate_fraud_risk(data, ranges):
    """Simulate a fraud check that takes longer for larger amounts."""
    amount = data['Amount']
    for (lower, upper), (loc, scale) in ranges:
        if lower <= amount < upper:
            return np.random.normal(loc=loc, scale=scale)


def collect_runtimes(data, sample_sizes, ranges):
    """Run the simulation at multiple sample sizes."""
    results = {}
    for size in sample_sizes:
        sample = data.sample(size)
        results[size] = [
            calculate_fraud_risk(row, ranges)
            for _, row in sample.iterrows()
        ]
    return results

Let's run the simulation on three sample sizes (100K, 1M, and the full dataset) for 
each data type.

In [ ]:
sample_sizes = [100_000, 1_000_000, len(real_data)]

real_runtimes = collect_runtimes(
    real_data, sample_sizes, ranges
)
fake_runtimes = collect_runtimes(
    fake_data, sample_sizes, ranges
)
synthetic_runtimes = collect_runtimes(
    synthetic_data, sample_sizes, ranges
)

Here are the runtime distributions at each sample size:

**Runtime Distributions for Real Data**

*The real data produces a multi-modal runtime distribution with distinct peaks 
corresponding to the amount-based processing tiers. The pattern is consistent 
across all three sample sizes, becoming sharper with more data.*

**Runtime Distributions for Fake Data**

**Runtime Distributions for Fake Data**

*The fake data produces a completely different shape — dominated by a single large 
peak around 0.7, with a small spike near 0.1. This is because the fake data's uniform 
amount distribution sends most transactions to the highest processing tier.*

**Runtime Distributions for Synthetic Data**

**Runtime Distributions for Synthetic Data**

*The synthetic data reproduces the same multi-modal pattern as the real data — the 
same peaks appear at the same positions, with similar relative heights. This confirms 
that preserving the amount distribution translates directly to realistic downstream behavior.*

The synthetic data produces runtime distributions that closely match the real data at 
every sample size. The fake data produces a fundamentally different shape because its 
uniform transaction amounts push most values into the highest processing tier.

## Quantifying Runtime Distribution Similarity

Using [KSComplement](https://docs.sdv.dev/sdmetrics/metrics/metrics-glossary/kscomplement), 
we can put a number on how similar the runtime distributions are. A score of 1.0 means the 
distributions are identical; a score of 0.0 means they are completely different.

**Fake vs. Real Data:**

In [ ]:
from sdmetrics.single_column import KSComplement

full_size = sample_sizes[2]

KSComplement.compute(
    real_data=pd.Series(real_runtimes[full_size]),
    synthetic_data=pd.Series(fake_runtimes[full_size])
)

**Synthetic vs. Real Data:**

In [ ]:
KSComplement.compute(
    real_data=pd.Series(real_runtimes[full_size]),
    synthetic_data=pd.Series(synthetic_runtimes[full_size])
)

> **Key Takeaway:** The synthetic data achieves a much higher KSComplement score (~0.86) than the fake data (~0.29), confirming that synthetic data preserves the runtime patterns from the real data.

## Comparing Total Runtimes

Finally, let's compare the total processing times across all three datasets and sample sizes.

In [ ]:
total_runtimes_df = pd.DataFrame({
    "Sample Size": sample_sizes * 3,
    "Total Runtime": [
        sum(real_runtimes[s]) for s in sample_sizes
    ] + [
        sum(fake_runtimes[s]) for s in sample_sizes
    ] + [
        sum(synthetic_runtimes[s]) for s in sample_sizes
    ],
    "Dataset Type": (
        ['Real'] * 3 + ['Fake'] * 3 + ['Synthetic'] * 3
    )
})

pd.options.display.float_format = '{:.2f}'.format
total_runtimes_df

*All three datasets scale linearly with sample size, but at different rates. 
The fake data (blue) has the highest total runtime because its uniform amounts 
overrepresent the expensive processing tiers. The synthetic data (teal) tracks 
closer to the real data (dark), confirming that realistic amount distributions 
lead to realistic aggregate performance.*

## Conclusion

In this tutorial, we compared real, fake, and synthetic data using a credit card transaction dataset.

| Metric | Fake Data | Synthetic Data |
|--------|-----------|----------------|
| Quality Score | Low | High |
| Distribution Match | Poor | Strong |
| Runtime Similarity (KSComplement) | ~0.29 | ~0.86 |

> **Key Takeaway:** Synthetic data is a far better alternative to fake data for testing, development, and simulation. It preserves the statistical patterns in your real data, which means downstream tasks produce realistic results.

To learn more, visit the [SDV documentation](https://docs.sdv.dev/sdv/).